In [ ]:
import numpy as np
import pandas as pd
from time import time
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from dirarte.datasets import FicoDataset, CreditDataset, GermanDataset, CompasDataset, BailDataset
from dirarte import FeatureTweakingExplainer, RobxExplainer, BetaRobustExplainer, DirarteExplainer

np.random.seed(0)

In [2]:
def run(
    dataset, 
    estimator='XGBoost', 
    shift_type='split',
    cost_type='tlps',
    n_iter=10,
    max_samples=-1,
):
    
    np.random.seed(0)
    results = {
        'dataset': [],
        'estimator': [],
        'shift_type': [],
        'cost_type': [],
        'n_samples': [],
        'method': [],
        'parameter': [],
        'validity (before)': [],
        'validity (after)': [],
        'cost': [],
        'plausibility': [],
        'sparsity': [],
        'time': [],     
        'time per samples': [],   
    }
    
    for i in tqdm(range(n_iter)):
        
        X_before, X_after, X_test, y_before, y_after, y_test = dataset.get_shifted_dataset(
            test_size=0.2, 
            shift_type=shift_type, 
            delete_fraction=0.1,
            label_shift_fraction=0.1
        )
        constraints = dataset.constraints

        if estimator == 'XGBoost':
            clf = XGBClassifier(n_estimators=100, max_depth=6).fit(X_before, y_before)
            clf_shift = XGBClassifier(n_estimators=100, max_depth=6).fit(X_after, y_after)
            prune_level = 1.0
        elif estimator == 'RandomForest':
            clf = RandomForestClassifier(n_estimators=100, max_depth=6).fit(X_before, y_before)
            clf_shift = RandomForestClassifier(n_estimators=100, max_depth=6).fit(X_after, y_after)
            prune_level = 0.5
        else:
            raise ValueError(f"Unsupported estimator: {estimator}")
            
        if max_samples > 0:
            X_target = X_test[clf.predict(X_test) == 0][:max_samples]
        else:
            X_target = X_test[clf.predict(X_test) == 0]
        
        for method in ['Baseline', 'RobX', 'BetaRobust', 'DiRARTE-W', 'DiRARTE-X']:
            
            if method == 'Baseline':
                explainer = FeatureTweakingExplainer(clf, constraints, cost_type=cost_type, prune_level=prune_level)
                explainer = explainer.initialize(X_before)
                time_baseline = time()
                recourse = explainer.explain_recourse(X_target)
                time_baseline = time() - time_baseline
                X_cf = recourse.counterfactual
                results['dataset'].append(dataset.name)
                results['estimator'].append(estimator)
                results['shift_type'].append(shift_type)
                results['cost_type'].append(cost_type)
                results['n_samples'].append(len(X_target))
                results['method'].append(method)
                results['parameter'].append(0.0)
                results['validity (before)'].append(clf.predict(X_cf).mean())
                results['validity (after)'].append(clf_shift.predict(X_cf).mean())
                results['cost'].append(recourse.get_average_cost())
                results['plausibility'].append(recourse.get_average_plausibility())
                results['sparsity'].append(recourse.get_average_sparsity())
                results['time'].append(time_baseline)
                results['time per samples'].append(time_baseline / len(X_target))

            elif method == 'RobX':
                if estimator == 'XGBoost':
                    taus = [0.5, 0.55, 0.6, 0.65, 0.7]
                else:
                    taus = [0.5, 0.525, 0.55, 0.575, 0.6]
                explainer = RobxExplainer(clf, constraints, cost_type=cost_type, prune_level=prune_level, tau=taus)
                explainer = explainer.initialize(X_before)
                time_robx = time()
                recourses = explainer.explain_recourse(X_target)
                time_robx = time() - time_robx
                for tau, recourse in zip(taus, recourses):
                    X_cf = recourse.counterfactual
                    results['dataset'].append(dataset.name)
                    results['estimator'].append(estimator)
                    results['shift_type'].append(shift_type)
                    results['cost_type'].append(cost_type)
                    results['n_samples'].append(len(X_target))
                    results['method'].append(method)
                    results['parameter'].append(tau)
                    results['validity (before)'].append(clf.predict(X_cf).mean())
                    results['validity (after)'].append(clf_shift.predict(X_cf).mean())
                    results['cost'].append(recourse.get_average_cost())
                    results['plausibility'].append(recourse.get_average_plausibility())
                    results['sparsity'].append(recourse.get_average_sparsity())
                    results['time'].append(time_robx)
                    results['time per samples'].append(time_robx / len(X_target))

            elif method == 'BetaRobust':
                deltas = [0.7, 0.75, 0.8, 0.85, 0.9]
                explainer = BetaRobustExplainer(clf, constraints, cost_type=cost_type, prune_level=prune_level, delta=deltas)
                explainer = explainer.initialize(X_before, y_before)
                time_beta = time()
                recourses = explainer.explain_recourse(X_target)
                time_beta = time() - time_beta
                for delta, recourse in zip(deltas, recourses):
                    X_cf = recourse.counterfactual
                    results['dataset'].append(dataset.name)
                    results['estimator'].append(estimator)
                    results['shift_type'].append(shift_type)
                    results['cost_type'].append(cost_type)
                    results['n_samples'].append(len(X_target))
                    results['method'].append(method)
                    results['parameter'].append(delta)
                    results['validity (before)'].append(clf.predict(X_cf).mean())
                    results['validity (after)'].append(clf_shift.predict(X_cf).mean())
                    results['cost'].append(recourse.get_average_cost())
                    results['plausibility'].append(recourse.get_average_plausibility())
                    results['sparsity'].append(recourse.get_average_sparsity())
                    results['time'].append(time_beta)
                    results['time per samples'].append(time_beta / len(X_target))

            else:
                if method == 'DiRARTE-W':
                    if estimator == 'XGBoost':
                        epsilons = [1.5, 2.0, 2.5, 3.0, 3.5]
                    else:
                        epsilons = [0.1, 0.2, 0.3, 0.4, 0.5]
                    explainer = DirarteExplainer(clf, constraints, cost_type=cost_type, prune_level=prune_level, distribution='wasserstein', epsilon=epsilons)
                else:
                    if estimator == 'XGBoost':
                        epsilons = [0.02, 0.04, 0.06, 0.08, 0.1]
                    else:
                        epsilons = [0.04, 0.08, 0.12, 0.16, 0.2]
                    explainer = DirarteExplainer(clf, constraints, cost_type=cost_type, prune_level=prune_level, distribution='chi-squared', epsilon=epsilons)
                explainer = explainer.initialize(X_before)
                time_dirarte = time()
                recourses = explainer.explain_recourse(X_target)
                time_dirarte = time() - time_dirarte
                for epsilon, recourse in zip(epsilons, recourses):
                    X_cf = recourse.counterfactual
                    results['dataset'].append(dataset.name)
                    results['estimator'].append(estimator)
                    results['shift_type'].append(shift_type)
                    results['cost_type'].append(cost_type)
                    results['n_samples'].append(len(X_target))
                    results['method'].append(method)
                    results['parameter'].append(epsilon)
                    results['validity (before)'].append(clf.predict(X_cf).mean())
                    results['validity (after)'].append(clf_shift.predict(X_cf).mean())
                    results['cost'].append(recourse.get_average_cost())
                    results['plausibility'].append(recourse.get_average_plausibility())
                    results['sparsity'].append(recourse.get_average_sparsity())
                    results['time'].append(time_dirarte)
                    results['time per samples'].append(time_dirarte / len(X_target))
                
    return pd.DataFrame(results)

In [ ]:
results_xg_tlps = []
for shift_type in ['split', 'delete', 'label']:
    for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
        result = run(
            dataset, 
            estimator='XGBoost', 
            shift_type=shift_type,
            cost_type='tlps',
            n_iter=10,
        )
        results_xg_tlps.append(result)

100%|██████████| 10/10 [1:47:05<00:00, 642.55s/it]


In [ ]:
results_rf_tlps = []
for shift_type in ['split', 'delete', 'label']:
    for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
        result = run(
            dataset, 
            estimator='RandomForest', 
            shift_type=shift_type,
            cost_type='tlps',
            n_iter=10,
        )
        results_rf_tlps.append(result)

100%|██████████| 10/10 [5:51:37<00:00, 2109.71s/it] 


In [ ]:
results_xg_std = []
for shift_type in ['split', 'delete', 'label']:
    for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
        result = run(
            dataset, 
            estimator='XGBoost', 
            shift_type=shift_type,
            cost_type='std',
            n_iter=10,
        )
        results_xg_std.append(result)

100%|██████████| 10/10 [1:40:47<00:00, 604.73s/it]


In [ ]:
results_rf_std = []
for shift_type in ['split', 'delete', 'label']:
    for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
        result = run(
            dataset, 
            estimator='RandomForest', 
            shift_type=shift_type,
            cost_type='std',
            n_iter=10,
        )
        results_rf_std.append(result)

In [7]:
results = pd.concat(results_xg_tlps + results_rf_tlps + results_xg_std + results_rf_std, ignore_index=True)
results.to_csv('./results/results_comparison.csv', index=False)